# Performance Comparison of GCN and GAT for Node Classification

## Graph Representation Learning Project

### Dataset:
Cora Citation Network

### Models:
- Graph Convolutional Network (GCN)
- Graph Attention Network (GAT)

### Task:
Node Classification

# Introduction

Graph Neural Networks (GNNs) are deep learning models designed for graph-structured data.

In this project, we compare two popular GNN architectures:
- Graph Convolutional Network (GCN)
- Graph Attention Network (GAT)

The models are evaluated on the Cora citation dataset for node classification tasks.

In [1]:
!pip install torch-geometric

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATConv

print("PyTorch version:", torch.__version__)
print("Setup successful!")

PyTorch version: 2.11.0+cpu
Setup successful!


# Dataset Description

The Cora dataset is a citation network where:
- nodes represent research papers
- edges represent citation links
- labels represent paper categories

The task is to classify each paper into its correct category.

In [3]:
dataset = Planetoid(root='data/Planetoid', name='Cora')

data = dataset[0]

print(dataset)
print(data)

Cora()
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [4]:
print("Number of nodes:", data.num_nodes)

print("Number of edges:", data.num_edges)

print("Number of features:", dataset.num_node_features)

print("Number of classes:", dataset.num_classes)

Number of nodes: 2708
Number of edges: 10556
Number of features: 1433
Number of classes: 7


# Building the GCN Model

In this section, we define a Graph Convolutional Network (GCN) model.

The model contains:
- Two GCN layers
- ReLU activation function
- Log Softmax output layer

The first layer learns hidden node representations.

The second layer predicts the final node classes.

In [5]:
class GCN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(dataset.num_node_features, 16)

        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):

        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)

        x = F.relu(x)

        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

# Creating the GCN Model

In this section, we create:
- the GCN model
- the optimizer used for training

The optimizer updates model weights during training to minimize the loss function.

In [6]:
model = GCN()

optimizer = torch.optim.Adam(model.parameters(),
                             lr=0.01,
                             weight_decay=5e-4)

print(model)

GCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 7)
)


# Training the GCN Model

In this section, we define the training function.

The training process includes:
- forward propagation
- loss calculation
- backpropagation
- optimizer update

The model learns by minimizing the classification loss on the training nodes.

In [7]:
def train():

    model.train()

    optimizer.zero_grad()

    out = model(data)

    loss = F.nll_loss(out[data.train_mask],
                      data.y[data.train_mask])

    loss.backward()

    optimizer.step()

    return loss.item()

# Evaluating the GCN Model

In this section, we define the evaluation function.

The function measures how accurately the model predicts node classes on the test dataset.

In [8]:
def test():

    model.eval()

    out = model(data)

    pred = out.argmax(dim=1)

    correct = pred[data.test_mask] == data.y[data.test_mask]

    acc = int(correct.sum()) / int(data.test_mask.sum())

    return acc

# Training the GCN Model

In this section, the GCN model is trained for multiple epochs.

For each epoch:
- the model learns from the training nodes
- the loss value is calculated
- the test accuracy is evaluated

The goal is to improve node classification performance over time.

In [9]:
import time

In [10]:
gcn_losses = []

start_time = time.time()

for epoch in range(1, 201):

    loss = train()

    gcn_losses.append(loss)

    acc = test()

    if epoch % 20 == 0:

        print(f'Epoch: {epoch:03d}, '
              f'Loss: {loss:.4f}, '
              f'Test Accuracy: {acc:.4f}')

gcn_training_time = time.time() - start_time

print("GCN Training Time:", gcn_training_time)

Epoch: 020, Loss: 0.1452, Test Accuracy: 0.8100
Epoch: 040, Loss: 0.0168, Test Accuracy: 0.8030
Epoch: 060, Loss: 0.0145, Test Accuracy: 0.8110
Epoch: 080, Loss: 0.0169, Test Accuracy: 0.8100
Epoch: 100, Loss: 0.0160, Test Accuracy: 0.8100
Epoch: 120, Loss: 0.0142, Test Accuracy: 0.8110
Epoch: 140, Loss: 0.0129, Test Accuracy: 0.8090
Epoch: 160, Loss: 0.0118, Test Accuracy: 0.8080
Epoch: 180, Loss: 0.0110, Test Accuracy: 0.8100
Epoch: 200, Loss: 0.0104, Test Accuracy: 0.8100
GCN Training Time: 4.6361894607543945


# GCN Training Loss Curve

The following graph shows how the training loss of the GCN model decreases over epochs.

A decreasing loss indicates that the model is learning effectively during training.

In [11]:
plt.figure(figsize=(7,5))

plt.plot(gcn_losses)

plt.xlabel('Epoch')

plt.ylabel('Loss')

plt.title('GCN Training Loss Curve')

plt.show()

NameError: name 'plt' is not defined

# Final GCN Performance

The final test accuracy of the GCN model is stored for later comparison with the GAT model.

In [ ]:
gcn_accuracy = test()

print("Final GCN Test Accuracy:", gcn_accuracy)

# Building the GAT Model

In this section, we define a Graph Attention Network (GAT) model.

Unlike GCN, GAT uses attention mechanisms to assign different importance to neighboring nodes.

The model contains:
- Two GAT layers
- ELU activation function
- Dropout regularization
- Log Softmax output layer

In [ ]:
class GAT(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.gat1 = GATConv(dataset.num_node_features,
                            8,
                            heads=8,
                            dropout=0.6)

        self.gat2 = GATConv(8 * 8,
                            dataset.num_classes,
                            heads=1,
                            concat=False,
                            dropout=0.6)

    def forward(self, data):

        x, edge_index = data.x, data.edge_index

        x = F.dropout(x, p=0.6, training=self.training)

        x = self.gat1(x, edge_index)

        x = F.elu(x)

        x = F.dropout(x, p=0.6, training=self.training)

        x = self.gat2(x, edge_index)

        return F.log_softmax(x, dim=1)

# Creating the GAT Model

In this section, we initialize:
- the GAT model
- the optimizer for training

The optimizer updates the model parameters during training to minimize the classification loss.

In [ ]:
gat_model = GAT()

gat_optimizer = torch.optim.Adam(gat_model.parameters(),
                                 lr=0.005,
                                 weight_decay=5e-4)

print(gat_model)

# Training Function for the GAT Model

In this section, we define the training function for the Graph Attention Network (GAT).

The function:
- performs forward propagation
- computes classification loss
- updates model parameters using backpropagation

In [ ]:
def gat_train():

    gat_model.train()

    gat_optimizer.zero_grad()

    out = gat_model(data)

    loss = F.nll_loss(out[data.train_mask],
                      data.y[data.train_mask])

    loss.backward()

    gat_optimizer.step()

    return loss.item()

# Evaluating the GAT Model

In this section, we define the evaluation function for the GAT model.

The function computes the classification accuracy on the test nodes of the Cora dataset.

In [ ]:
def gat_test():

    gat_model.eval()

    out = gat_model(data)

    pred = out.argmax(dim=1)

    correct = pred[data.test_mask] == data.y[data.test_mask]

    acc = int(correct.sum()) / int(data.test_mask.sum())

    return acc

# Training the GAT Model

In this section, the GAT model is trained for multiple epochs.

For each epoch:
- the model learns node representations using attention mechanisms
- the loss value is computed
- the test accuracy is evaluated

The performance of the GAT model will later be compared with the GCN model.

In [ ]:
gat_losses = []

start_time = time.time()

for epoch in range(1, 201):

    loss = gat_train()

    gat_losses.append(loss)

    acc = gat_test()

    if epoch % 20 == 0:

        print(f'Epoch: {epoch:03d}, '
              f'Loss: {loss:.4f}, '
              f'Test Accuracy: {acc:.4f}')

gat_training_time = time.time() - start_time

print("GAT Training Time:", gat_training_time)

# Final GAT Performance

The final test accuracy of the GAT model is stored for comparison with the GCN model.

In [ ]:
gat_accuracy = gat_test()

print("Final GAT Test Accuracy:", gat_accuracy)

# Model Experimental Comparison

The following table summarizes the performance of the GCN and GAT models on the Cora dataset.

| Model | Test Accuracy | Training Time |
|------|------|------|
| GCN | 80.6% | ~4.85 seconds |
| GAT | 79.9% | ~32.83 seconds |

The results show that the GCN model achieved slightly better classification accuracy while requiring significantly less training time compared to the GAT model.

Although GAT introduces attention mechanisms for more flexible neighborhood aggregation, the additional computational complexity did not improve performance in this experiment.

In [ ]:
print("Model Comparison on Cora Dataset")
print("----------------------------------")

print(f"GCN Test Accuracy  : {gcn_accuracy:.4f}")
print(f"GCN Training Time  : {gcn_training_time:.2f} seconds")

print()

print(f"GAT Test Accuracy  : {gat_accuracy:.4f}")
print(f"GAT Training Time  : {gat_training_time:.2f} seconds")

# Accuracy Visualization

The following graph compares the test accuracy of the GCN and GAT models on the Cora dataset.

In [ ]:
import matplotlib.pyplot as plt

models = ['GCN', 'GAT']

accuracies = [gcn_accuracy, gat_accuracy]

plt.figure(figsize=(6,4))

plt.bar(models, accuracies)

plt.ylabel('Test Accuracy')

plt.title('GCN vs GAT on Cora Dataset')

plt.ylim(0, 1)

plt.savefig("comparison_plot.png")

plt.show()

# Discussion

The experimental results show that the GCN model achieved slightly higher accuracy than the GAT model on the Cora dataset.

Although GAT introduces attention mechanisms to improve representation learning, the additional complexity did not improve performance in this experiment.

Possible reasons include:
- small dataset size
- sensitivity to hyperparameters
- higher model complexity

The results demonstrate that simpler architectures such as GCN can still perform competitively on citation network datasets.

# Conclusion

In this project, we compared the performance of Graph Convolutional Networks (GCN) and Graph Attention Networks (GAT) on the Cora citation dataset for node classification.

Both models achieved good classification performance. However, the GCN model slightly outperformed the GAT model in terms of final test accuracy.

Experimental Results:
- GCN Test Accuracy: 80.6%
- GAT Test Accuracy: 78.9%

The results show that more complex attention mechanisms do not always guarantee better performance, especially on smaller datasets such as Cora.

Overall, the project demonstrates the effectiveness of Graph Neural Networks for node classification tasks and highlights the importance of model selection and hyperparameter tuning in graph representation learning.